In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
from datetime import datetime, timedelta
import re
import time
from deep_translator import GoogleTranslator


In [ ]:
driver = webdriver.Chrome()
driver.get("LINK") #add link
wait = WebDriverWait(driver, 10)
print(driver.title)

In [ ]:

def close_popup():
    try:
        popup = driver.find_element(By.ID, "educationalPopup")

        if popup.is_displayed():
            # Look for the close button
            buttons = popup.find_elements(
                By.CSS_SELECTOR,
                'button.close, button.btn-close, '
                '[data-dismiss="modal"], [data-bs-dismiss="modal"]'
            )

            if buttons:
                driver.execute_script(
                    "arguments[0].click();", buttons[0]
                )
            else:
                # Try the Escape key if no button is found
                from selenium.webdriver.common.keys import Keys
                driver.find_element(By.TAG_NAME, "body").send_keys(
                    Keys.ESCAPE
                )

            WebDriverWait(driver, 5).until(
                EC.invisibility_of_element_located(
                    (By.ID, "educationalPopup")
                )
            )

            print("Popup closed.")

    except TimeoutException:
        print("The popup is still visible.")
    except Exception as e:
        print("Popup check:", e)

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

wait = WebDriverWait(driver, 20)

tenders = []
seen_tenders = set()


# --------------------------------------------------
# Close the educational popup
# --------------------------------------------------

def close_popup():
    try:
        popup = driver.find_element(By.ID, "educationalPopup")

        if not popup.is_displayed():
            return

        buttons = popup.find_elements(
            By.CSS_SELECTOR,
            'button.close, button.btn-close, '
            '[data-dismiss="modal"], [data-bs-dismiss="modal"]'
        )

        if buttons:
            driver.execute_script(
                "arguments[0].click();", buttons[0]
            )
        else:
            driver.find_element(
                By.TAG_NAME, "body"
            ).send_keys(Keys.ESCAPE)

        wait.until(
            EC.invisibility_of_element_located(
                (By.ID, "educationalPopup")
            )
        )

        print("Popup closed.")

    except NoSuchElementException:
        pass

    except TimeoutException:
        # Last resort: try Escape
        driver.find_element(
            By.TAG_NAME, "body"
        ).send_keys(Keys.ESCAPE)

        wait.until(
            EC.invisibility_of_element_located(
                (By.ID, "educationalPopup")
            )
        )


# --------------------------------------------------
# Click an element safely
# --------------------------------------------------

def click_element(element):
    close_popup()

    try:
        wait.until(
            lambda d: element.is_displayed()
            and element.is_enabled()
        )

        element.click()

    except ElementClickInterceptedException:
        close_popup()

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            element
        )

        time.sleep(0.3)

        try:
            element.click()
        except ElementClickInterceptedException:
            # Last resort
            driver.execute_script(
                "arguments[0].click();", element
            )


# --------------------------------------------------
# Convert the expiration date
# --------------------------------------------------

def convert_expiration_date(expiration_date):
    match = re.search(
        r'in\s+(\d+)\s+days',
        expiration_date,
        re.IGNORECASE
    )

    if match:
        days_ago = int(match.group(1))

        return (
            datetime.today() + timedelta(days=days_ago)
        ).strftime("%d/%m/%Y")

    return "Inconnu"


# --------------------------------------------------
# Process the current page
# --------------------------------------------------

def process_page():
    close_popup()

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "#tendeList tbody tr")
        )
    )

    rows = driver.find_elements(
        By.CSS_SELECTOR, "#tendeList tbody tr"
    )

    i = 0

    while i < len(rows):
        try:
            close_popup()

            rows = driver.find_elements(
                By.CSS_SELECTOR, "#tendeList tbody tr"
            )

            if i >= len(rows):
                break

            row = rows[i]

            cells = row.find_elements(By.TAG_NAME, "td")

            # Skip detail rows and invalid rows
            if len(cells) < 6:
                i += 1
                continue

            title = cells[2].text.strip()
            publish_date = cells[4].text.strip()
            expiration_date = cells[5].text.strip()

            if not title:
                i += 1
                continue

            # Avoid duplicates
            dedup_key = (title, publish_date)

            if dedup_key in seen_tenders:
                i += 1
                continue

            actual_date = convert_expiration_date(
                expiration_date
            )

            link = "Lien non trouvé"

            # ------------------------------------------
            # Open the detail row
            # ------------------------------------------

            try:
                control = row.find_element(
                    By.CSS_SELECTOR, "td.details-control"
                )

                click_element(control)

                # DataTables normally inserts a row
                # of details immediately after the main row
                detail_row = wait.until(
                    lambda d: (
                        d.find_elements(
                            By.CSS_SELECTOR,
                            "#tendeList tbody tr"
                        )[i + 1]
                        if len(
                            d.find_elements(
                                By.CSS_SELECTOR,
                                "#tendeList tbody tr"
                            )
                        ) > i + 1
                        else False
                    )
                )

                pdf_links = detail_row.find_elements(
                    By.CSS_SELECTOR,
                    'a[href*=".pdf"]'
                )

                if pdf_links:
                    link = pdf_links[0].get_attribute("href")

                # Close the details before the next row
                rows = driver.find_elements(
                    By.CSS_SELECTOR, "#tendeList tbody tr"
                )

                if i < len(rows):
                    control = rows[i].find_element(
                        By.CSS_SELECTOR, "td.details-control"
                    )

                    click_element(control)

            except (
                TimeoutException,
                NoSuchElementException,
                StaleElementReferenceException,
                IndexError,
                ElementClickInterceptedException
            ) as e:
                print(
                    f"Unable to retrieve the PDF "
                    f"for '{title}': {e}"
                )

            # ------------------------------------------
            # Save the tender without translation
            # ------------------------------------------

            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": actual_date,
                "link": link,
                "country": "South Africa"
            })

            seen_tenders.add(dedup_key)

            print(f"Tender processed: {title}")
            print(f"PDF link: {link}")

        except StaleElementReferenceException:
            print(
                f"Row {i} became stale. "
                "Refreshing the rows."
            )

        except Exception as e:
            print(f"Error on row {i}: {e}")

        i += 1

    print(f"Page processed. Total collected: {len(tenders)}")


# --------------------------------------------------
# Go through all the pages
# --------------------------------------------------

while True:
    process_page()

    try:
        close_popup()

        next_button = wait.until(
            EC.presence_of_element_located(
                (By.ID, "tendeList_next")
            )
        )

        classes = next_button.get_attribute("class") or ""

        if "disabled" in classes:
            print("End of pagination.")
            break

        rows_before = driver.find_elements(
            By.CSS_SELECTOR, "#tendeList tbody tr"
        )

        if not rows_before:
            print("No row found.")
            break

        first_line = rows_before[0]

        click_element(next_button)

        wait.until(
            EC.staleness_of(first_line)
        )

        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#tendeList tbody tr")
            )
        )

        time.sleep(0.5)

    except TimeoutException:
        print("End of pagination or timeout.")
        break

    except Exception as e:
        print(f"Pagination error: {e}")
        break


# --------------------------------------------------
# Final result
# --------------------------------------------------

import pandas as pd

df_tenders = pd.DataFrame(tenders)

print("\nCollection finished.")
print("Total number of tenders:", len(df_tenders))

display(df_tenders)

In [ ]:
driver.quit()

In [ ]:
def clean_excel_string(s):
    if isinstance(s, str):
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

df = pd.DataFrame(tenders)
df = df.applymap(clean_excel_string)

# save file
df.to_excel("../2_scraped_files/SouthAfrica2.xlsx", index=False) # modify